In [1]:
import fitz
import os
import io
import pandas as pd
from PIL import Image
from pathlib import Path

In [2]:
PDF_FOLDER = Path("../data/input_pdfs")
OUTPUT_FOLDER = Path("../data/rendered_pages")
METADATA_FOLDER = Path("../data/metadata")
CSV_PATH = METADATA_FOLDER / "rendered_pages.csv"

DPI = 300
ZOOM = DPI / 72
print("Rendering at", DPI, "DPI")

Rendering at 300 DPI


In [3]:
def render_page(page, dpi=300):
    """
    Render a PDF page to a PIL Image.
    """

    zoom = dpi / 72

    matrix = fitz.Matrix(zoom, zoom)

    pix = page.get_pixmap(matrix=matrix, alpha=False)

    img = Image.frombytes(
        "RGB",
        [pix.width, pix.height],
        pix.samples
    )

    return img

In [4]:
pdf_files = sorted(PDF_FOLDER.glob("*.pdf"))

print(f"{len(pdf_files)} PDF(s) found")

for pdf in pdf_files:
    print(pdf.name)

3 PDF(s) found
sample_pdf_1.pdf
sample_pdf_2.pdf
sample_pdf_3.pdf


In [5]:
records = []

for pdf_path in pdf_files:

    pdf_name = pdf_path.stem

    output_pdf_folder = OUTPUT_FOLDER / pdf_name

    output_pdf_folder.mkdir(exist_ok=True)

    doc = fitz.open(pdf_path)

    print(f"\nProcessing {pdf_name}")

    for page_number in range(len(doc)):

        page = doc.load_page(page_number)

        image = render_page(page, dpi=DPI)

        filename = f"{pdf_name}_page_{page_number+1:03d}.png"

        filepath = output_pdf_folder / filename

        image.save(filepath)

        records.append({

            "pdf_name": pdf_name,

            "page_number": page_number + 1,

            "filename": filename,

            "filepath": str(filepath),

            "width": image.width,

            "height": image.height,

            "dpi": DPI,

            "chemical_detected": None,

            "cropped": False,

            "decimer_result": None

        })

    doc.close()


Processing sample_pdf_1

Processing sample_pdf_2

Processing sample_pdf_3


In [6]:
df = pd.DataFrame(records)

print(df.head())

print()

print("Total Rendered Pages:", len(df))

       pdf_name  page_number                   filename  \
0  sample_pdf_1            1  sample_pdf_1_page_001.png   
1  sample_pdf_1            2  sample_pdf_1_page_002.png   
2  sample_pdf_1            3  sample_pdf_1_page_003.png   
3  sample_pdf_1            4  sample_pdf_1_page_004.png   
4  sample_pdf_1            5  sample_pdf_1_page_005.png   

                                            filepath  width  height  dpi  \
0  ..\data\rendered_pages\sample_pdf_1\sample_pdf...   2401    3001  300   
1  ..\data\rendered_pages\sample_pdf_1\sample_pdf...   2400    3000  300   
2  ..\data\rendered_pages\sample_pdf_1\sample_pdf...   2400    3000  300   
3  ..\data\rendered_pages\sample_pdf_1\sample_pdf...   2400    3000  300   
4  ..\data\rendered_pages\sample_pdf_1\sample_pdf...   2400    3000  300   

  chemical_detected  cropped decimer_result  
0              None    False           None  
1              None    False           None  
2              None    False           None  
3   

In [7]:
display(df.groupby("pdf_name").size().reset_index(name="pages"))

,pdf_name,pages
0,sample_pdf_1,8
1,sample_pdf_2,10
2,sample_pdf_3,3


In [8]:
df.to_csv(CSV_PATH, index=False)
print("Metadata saved to")
print(CSV_PATH)

Metadata saved to
..\data\metadata\rendered_pages.csv
